# Extra. 저장된 LoRA Adapter 재사용하기

- `trainser.train()`을 통해 학습하는 과정은 꽤나 오랜 시간이 소요됨
- 특히, 현재로서는 기껏 학습한 모델이 세션이 종료되면 즉시 사라지게 됨.
- 즉, 매번 새로운 세션을 열 때마다 다시 학습해야 하는 번거로움이 있음.
- 따라서, 한번 **미리 학습을 완료 한** 결과물인 `LoRA 어댑터` 파일을 저장하여 사용할 수 있음

## E-1. 왜 `어댑터`만 저장해서 사용하는가?

1. **병합 모델**
    - 이 실습에서 사용된 모델 약 2.5GB + LoRA 어댑터(수백 MB)를 공유할 수는 있음.
    - 단, 결과물 용량이 수 GB에 달하므로 공유과정이 매우 비효율 적임
2. **LoRA 어댑터**
    - 오랜 시간 학습한 `변화량`만 별도로 저장
    - 용량이 상대적으로 낮으며, 이렇게 만들어진 LoRA 어댑터를 다른 환경에서 동일한 모델에 부착하는 것은 어려운 과정이 아님

## E-2. LoRA 어댑터 저장하기

1. **방법 1:** `save_pretrained`
    - `trainer.train()` 과정이 종료된 직후, base_model의 메모리에는 최종 학습된 어댑터가 로드되어 있음.
    - 이 어댑터를 새로운 폴더에 저장하는 방법

    ```python
      # "my_lora_adapter"라는 폴더에 학습된 '어댑터'만 저장
      base_model.save_pretrained("my_lora_adapter")

      # (선택) 토크나이저도 함께 저장
      tokenizer.save_pretrained("my_lora_adapter")
    ```

2. **방법 2:** `output_dir`
    - `SFTTrainer` 작성 시, `TrainingArguments`에서 설정한 `output_dir = "outputs”` 을 통해 저장
    - 모델이 학습되는 과정을 지정한 폴더에 저장하는 명령어
    - 일정 주기로 학습된 결과는 일반적으로 `outputs/checkpoint-학습횟수`  형태의 폴더로 저장
    - 이중, 가장 마지막 폴더의 내용이 학습된 **LoRA 어댑터 폴더**
    - 단, 위처럼 tokenizer 역시 공유하고자 한다면, 별도로 복사하여 사용
    
    ```python
    # (예시) 학습 완료 후, 마지막 체크포인트 폴더를 'my_lora_adapter'로 이름 변경
    !mv /content/outputs/checkpoint-1251 /content/my_lora_adapter
    
    # (선택) 토크나이저도 폴더에 함께 저장
    tokenizer.save_pretrained("my_lora_adapter")
    ```

In [5]:
#!unzip my_lora_adapter.zip
#!ls -F


# 1. 'my_lora_adapter' 이름의 새 폴더 생성
!mkdir my_lora_adapter

# 2. 압축이 풀린 핵심 어댑터 파일들을 새 폴더로 이동 (cp 대신 mv 사용)
# *.json 파일들 이동
!mv *.json my_lora_adapter/

# *.safetensors (또는 *.bin) 파일 이동 - 파일 이름이 다를 경우 수정하세요
!mv *.safetensors my_lora_adapter/

## E-3. 어댑터 활용

### E-3-1. 환경 설치

- 4비트 양자화 모델을 사용할 것이므로, unsloth 설치 등 기본 환경 설정은 기존과 동일

In [1]:
import re
import torch
v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
!pip install --no-deps bitsandbytes==0.45.5 accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install -U bitsandbytes

  Using cached bitsandbytes-0.45.5-py3-none-manylinux_2_24_x86_64.whl.metadata (5.0 kB)
Using cached bitsandbytes-0.45.5-py3-none-manylinux_2_24_x86_64.whl (76.1 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.48.1
    Uninstalling bitsandbytes-0.48.1:
      Successfully uninstalled bitsandbytes-0.48.1
  Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl (60.1 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.45.5
    Uninstalling bitsandbytes-0.45.5:
      Successfully uninstalled bitsandbytes-0.45.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.10.8 requires tyro, which is not installed.
unsloth 2025.10.8 requires trl!=0.15.0,!=0.19.0,!=0.9.0,!=0.9.1,!=0.9.2,!=0.9.3

### E-3-2. 모델 로딩 및 어댑터 활용

1. 모델 로딩 방식은 기존과 동일
2. 공유 받은 어댑터를 모델과 동일한 위치에 두고, 불러오기
    - google drive 활용

In [2]:
# 구글 드라이브를 코랩 환경에 마운트.
from google.colab import drive
drive.mount('/content/drive')

# API 키 파일이 저장된 기본 경로를 설정.
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/10_QLoRA/'

Mounted at /content/drive


In [6]:
from unsloth import FastLanguageModel

# 4비트 베이스 모델 로드
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# '어댑터'를 베이스 모델에 부착 (Adapter Injection)
base_model.load_adapter("my_lora_adapter")
# base_model.load_adapter(base_path + "my_lora_adapter")

==((====))==  Unsloth 2025.10.8: Fast Mistral patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### E-3-3. 추론 실행

In [7]:
from transformers import TextStreamer

# 1. 테스트할 문장 준비
english_input = 'Acephala have no head and instead have well-developed feet that are directly attached to the torso.'
korean_answer = '무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.'

test_messages = [
    {"role": "system", "content": "You are an expert translator. Translate the user's English text into Korean."},
    {"role": "user", "content": english_input}
]

# 2. 모델 입력값(텐서) 생성 (추론이므로 add_generation_prompt = True)
chat_string = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True,
    tokenize = False
)
inputs = tokenizer(
    chat_string,
    return_tensors="pt"
).to(base_model.device)

# 3. 추론 실행 (학습된 모델이 대답)
print(f"입력: {english_input}")
print(f"정답 예시: {korean_answer}")
print('예측: ')
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = base_model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128, # 최대 128 토큰 생성
    use_cache = True
)

입력: Acephala have no head and instead have well-developed feet that are directly attached to the torso.
정답 예시: 무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.
예측: 
무두추는 머리가 없고, 대신 몸통에 직접 붙어 있는 발근이 잘 발달되어 있습니다.<|end|>


In [8]:
from transformers import TextStreamer

# 1. 테스트할 문장 준비
english_input = 'he rapid advancement of artificial intelligence is fundamentally changing how we approach complex problems in various industries.'
korean_answer = '인공지능의 급속한 발전은 다양한 산업에서 우리가 복잡한 문제에 접근하는 방식을 근본적으로 변화시키고 있습니다.'
test_messages = [
    {"role": "system", "content": "You are an expert translator. Translate the user's English text into Korean."},
    {"role": "user", "content": english_input}
]

# 2. 모델 입력값(텐서) 생성 (추론이므로 add_generation_prompt = True)
chat_string = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True,
    tokenize = False
)
inputs = tokenizer(
    chat_string,
    return_tensors="pt"
).to(base_model.device)

# 3. 추론 실행 (학습된 모델이 대답)
print(f"입력: {english_input}")
print(f"정답 예시: {korean_answer}")
print('예측: ')
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = base_model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128, # 최대 128 토큰 생성
    use_cache = True
)

입력: he rapid advancement of artificial intelligence is fundamentally changing how we approach complex problems in various industries.
정답 예시: 인공지능의 급속한 발전은 다양한 산업에서 우리가 복잡한 문제에 접근하는 방식을 근본적으로 변화시키고 있습니다.
예측: 
인공지능의 급속한 발전은 다양한 산업에서 우리가 처리하는 복잡한 문제에 대한 접근 방식을 바꾸고 있습니다.<|end|>
